# WISE dataset evaluation

Reproduce declared assessments and inspect native WISE priorities. The collection contains 15 configurations and 44 primary views; overlapping exports and the three-case workbook are not independent replications. Norms are illustrative, not stakeholder-approved policies.

Start a kernel in the environment described in [the evaluation guide](../README.md). Raw inputs and prepared outputs remain local. This notebook executes native replay by default; set `REBUILD_RAW = True` to regenerate assessments from catalogue-verified inputs first.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import wise

candidates = [Path(os.environ["WISE_EVALUATION_ROOT"])] if "WISE_EVALUATION_ROOT" in os.environ else []
for ancestor in (Path.cwd(), *Path.cwd().parents):
    candidates.extend([ancestor, ancestor / "packages/wise-pm/evaluation"])
EVALUATION = next((p.resolve() for p in candidates if (p / "data_catalog.json").is_file()), None)
if EVALUATION is None:
    raise FileNotFoundError("Open this notebook inside the repository, or set WISE_EVALUATION_ROOT")
sys.path.insert(0, str(EVALUATION))
from data_manager import DEFAULT_DATA_ROOT, load_catalogue, verify_data
from native_assessments import carrier_log

REBUILD_RAW = False
DATASET = "bpic2013_incidents"
VIEW = "Balanced"
PREPARED = EVALUATION / "data/prepared"
RUN = EVALUATION / "results" / datetime.now(timezone.utc).strftime("notebook-%Y%m%dT%H%M%S%fZ")
print("wise", wise.__version__, "| configuration:", DATASET, "| view:", VIEW)

## Inspect source identities and frozen expectations

The catalogue distinguishes source families from scored configurations. Hash verification is required for a raw replay. A prepared replay does not require raw archives and does not attest that raw extraction was rerun. The expectation table reports observation coverage; it does not rank different datasets against one another.

In [ ]:
catalogue = load_catalogue(EVALUATION / "data_catalog.json")
expected = json.loads((EVALUATION / "expected/native_summary.json").read_text())
display(pd.Series({k: expected[k] for k in ["configurations", "primary_views", "max_residual", "absolute_tolerance"]}))
summary = pd.read_csv(EVALUATION / "expected/cross_domain_summary.csv")
display(summary[["dataset", "view", "input_cases", "mean_weighted_coverage", "evidence_origin"]])

## Execute the pipeline

Every invocation creates a new result directory and preserves earlier runs. Raw mode verifies exact input bytes, executes all adapters and their witnesses, and then calls the installed library. Prepared mode reuses the locally stored assessments. Both exercise serialized norms, native scores and priorities, signed layer explanations, and differential scoring checks.

In [ ]:
command = [sys.executable, str(EVALUATION / "run.py"), "--output", str(RUN)]
if REBUILD_RAW:
    display(verify_data(DEFAULT_DATA_ROOT, catalogue))
else:
    if not PREPARED.is_dir():
        raise FileNotFoundError("Prepared inputs are missing. Import the raw catalogue and set REBUILD_RAW = True.")
    command.extend(["--prepared-root", str(PREPARED)])
subprocess.run(command, cwd=EVALUATION, check=True)
execution = json.loads((RUN / "run_manifest.json").read_text())
verification = json.loads((RUN / "native/manifest.json").read_text())
assert execution["status"] == verification["status"] == "passed"
display(pd.Series({k: verification[k] for k in ["status", "datasets", "views", "case_applications", "max_residual"]}))

## Load one norm and call the public library

The raw adapter has already assigned bounded criterion severities and scope flags. These are transported as case attributes into identity `Metric` constraints. This verifies native aggregation after adaptation; the single assessment record per case is not a source process trace. Inspect raw witnesses for event evidence.

In [ ]:
dataset_dir = RUN / "native" / DATASET
features = pd.read_parquet(dataset_dir / "assessment_features.parquet")
norm = wise.Norm.load(EVALUATION / "norms/native" / f"{DATASET}.json")
result = wise.score(carrier_log(features), norm, derive=False)
backlog = wise.prioritize(result, by="assessment_group", view=VIEW, gamma=0)
display(backlog.head(10))
assert result.scores.index.equals(features.index)
assert np.allclose(
    result.scores[VIEW], pd.read_parquet(dataset_dir / "native_case_scores.parquet")[f"score__{VIEW}"], equal_nan=True
)

## Explain a priority without losing negative offsets

Native layer-driver deltas are differences from the overall mean penalty. Multiply them by scored group support to obtain signed excess contributions. Negative contributions offset positive ones; only the final sum is clipped at zero to reproduce the unshrunk priority index. These are explanatory accounting terms, not causal effects or estimated savings.

In [ ]:
drivers = wise.layer_drivers(result, by="assessment_group", view=VIEW)
signed = pd.DataFrame({layer: drivers[f"{layer}__delta"] * drivers["n_cases"] for layer in norm.layer_ids})
assert np.allclose(signed.sum(axis=1).clip(lower=0).reindex(backlog.index), backlog["PI"])
group = backlog.index[0]
values = signed.loc[group].sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(values.index, values.to_numpy(), color=["#3f7f93" if x < 0 else "#ce6c3e" for x in values])
ax.axvline(0, color="#444444", linewidth=0.8)
ax.set(title=f"{DATASET}: {group} — {VIEW}", xlabel="Signed contribution to excess penalty mass")
fig.tight_layout()
plt.show()
print("Priority:", float(backlog.loc[group, "PI"]))

## Interpret the evidence

A high priority identifies a group for investigation under the declared norm. Group membership does not establish responsibility. Missing evidence remains distinct from an observed absence of a violation; zero observed concern does not demonstrate complete observation. Read the adapter configuration, source catalogue and per-configuration verification together.

For BPIC13 open problems, missing team fields lower coverage even when two view rankings agree. In the AMR data, duplicate handling changes the leading group. The simulated OCEL log contains dangling object relationships, and the healthcare rules describe recorded process patterns without making clinical judgments. These limitations remain part of the result even when numerical checks pass.

Change `DATASET` and `VIEW` to inspect another declared configuration. Any change to extraction rules, thresholds, scope, weights or grouping needs a separately identified rerun; do not reuse frozen results as evidence for a modified norm.